In [2]:
# ChatGPT code

In [4]:
from collections import deque


class MinCostMaxFlow:
    def __init__(self, n):
        self.n = n
        self.graph = [[] for _ in range(n)]

    def add_edge(self, u, v, capacity, cost):
        """
        添加一条 u -> v 的边：
        capacity: 容量
        cost: 单位费用
        """
        forward = [v, capacity, cost, None]
        backward = [u, 0, -cost, None]

        forward[3] = len(self.graph[v])
        backward[3] = len(self.graph[u])

        self.graph[u].append(forward)
        self.graph[v].append(backward)

    def min_cost_max_flow(self, source, sink):
        total_flow = 0
        total_cost = 0
        INF = float("inf")

        while True:
            dist = [INF] * self.n
            in_queue = [False] * self.n
            parent_node = [-1] * self.n
            parent_edge = [-1] * self.n

            dist[source] = 0
            queue = deque([source])
            in_queue[source] = True

            # SPFA：在残量网络中找最短路
            while queue:
                u = queue.popleft()
                in_queue[u] = False

                for i, edge in enumerate(self.graph[u]):
                    v, capacity, cost, rev = edge

                    if capacity > 0 and dist[v] > dist[u] + cost:
                        dist[v] = dist[u] + cost
                        parent_node[v] = u
                        parent_edge[v] = i

                        if not in_queue[v]:
                            queue.append(v)
                            in_queue[v] = True

            # 已经不存在从 source 到 sink 的可行路径
            if dist[sink] == INF:
                break

            # 找到本次可增广的最大流量
            augment = INF
            v = sink

            while v != source:
                u = parent_node[v]
                edge_index = parent_edge[v]
                augment = min(augment, self.graph[u][edge_index][1])
                v = u

            # 更新正向边、反向边
            v = sink
            while v != source:
                u = parent_node[v]
                edge_index = parent_edge[v]

                edge = self.graph[u][edge_index]
                rev_index = edge[3]

                edge[1] -= augment
                self.graph[v][rev_index][1] += augment

                v = u

            total_flow += augment
            total_cost += augment * dist[sink]

        return total_flow, total_cost

In [3]:
import networkx as nx

# 1. 创建一个有向图
G = nx.DiGraph()

# 2. 添加边，必须包含 capacity (容量) 和 weight (单位费用) 两个属性
# 数据格式: (起点, 终点, {'capacity': 容量, 'weight': 单位费用})
edges = [
    ('S', 'A', {'capacity': 4, 'weight': 2}),
    ('S', 'B', {'capacity': 3, 'weight': 5}),
    ('A', 'B', {'capacity': 2, 'weight': 1}),
    ('A', 'T', {'capacity': 3, 'weight': 6}),
    ('B', 'T', {'capacity': 4, 'weight': 2})
]
G.add_edges_from(edges)

# 3. 设定源点和汇点
source = 'S'
sink = 'T'

# 4. 调用 networkx 的 min_cost_flow 函数
# 注意：networkx 默认的 min_cost_flow 是解决需求满足问题的。
# 为了求单纯的“最小费用最大流”，我们需要使用 max_flow_min_cost 函数
flow_dict = nx.max_flow_min_cost(G, source, sink)

# 5. 计算最大流量和最小总费用
# flow_dict 是一个嵌套字典，记录了每条边实际流过的流量
max_flow_value = sum(flow_dict[source][v] for v in flow_dict[source])
min_cost_value = nx.cost_of_flow(G, flow_dict)

print(f"最大流量: {max_flow_value}")
print(f"在达到最大流时的最小总费用: {min_cost_value}")
print("\n每条边的实际流量分配方案:")
for u in flow_dict:
    for v, flow in flow_dict[u].items():
        if flow > 0: # 只打印有流量通过的边
            print(f"  {u} -> {v}: 流量 = {flow}")

最大流量: 7
在达到最大流时的最小总费用: 50

每条边的实际流量分配方案:
  S -> A: 流量 = 4
  S -> B: 流量 = 3
  A -> B: 流量 = 1
  A -> T: 流量 = 3
  B -> T: 流量 = 4


In [5]:
# 节点编号
# 0: S
# 1: 仓库A
# 2: 仓库B
# 3: 灾区1
# 4: 灾区2
# 5: 灾区3
# 6: T

S, A, B, D1, D2, D3, T = range(7)

mcmf = MinCostMaxFlow(7)

# 源点 -> 仓库：仓库供应量
mcmf.add_edge(S, A, 8, 0)
mcmf.add_edge(S, B, 7, 0)

# 仓库 -> 灾区：运输容量与单位费用
mcmf.add_edge(A, D1, 4, 2)
mcmf.add_edge(A, D2, 6, 4)
mcmf.add_edge(A, D3, 5, 5)

mcmf.add_edge(B, D1, 4, 3)
mcmf.add_edge(B, D2, 6, 1)
mcmf.add_edge(B, D3, 5, 2)

# 灾区 -> 汇点：灾区需求量
mcmf.add_edge(D1, T, 4, 0)
mcmf.add_edge(D2, T, 6, 0)
mcmf.add_edge(D3, T, 5, 0)

max_flow, min_cost = mcmf.min_cost_max_flow(S, T)

print("最大配送量：", max_flow)
print("最小总运输成本：", min_cost)

最大配送量： 15
最小总运输成本： 36


In [6]:
# 添加 A -> D1 前，记录该边在邻接表中的位置
edge_index = len(mcmf.graph[A])
mcmf.add_edge(A, D1, 4, 2)

# 求解后：
remaining_capacity = mcmf.graph[A][edge_index][1]
actual_flow = 4 - remaining_capacity

print("A 运往灾区1：", actual_flow)

A 运往灾区1： 0
